# RECIPE SIG: AUDIT DAN VALIDASI TOPOLOGI DATA VEKTOR BATAS ADMINISTRASI

**Nama:** DEVEN  
**NIM:** H061251013
**Kelas:** A  
 

---

## Identitas Recipe

| Item | Keterangan |
|---|---|
| **Recipe ID** | `R-GIS-001` |
| **Prompt ID** | `P-GIS-001` |
| **Bidang** | Sistem Informasi Geografis / Analisis Data Vektor |
| **Satu outcome utama** | Laporan QA/QC dan validasi topologi data vektor (*GeoJSON*/*Shapefile*): error geometri ditandai beserta alasannya tanpa menghapus/mengubah fitur asli |
| **Bukan outcome recipe ini** | Interpolasi spasial, *overlay/buffering* kompleks, pemodelan AHP, atau penyusunan *layouting* peta |
| **Status evidence** | **Executed** pada data spasial uji coba (lihat bagian *Run Record*) |
| **Artifact yang dihasilkan** | `batas_admin_qc.geojson`, `ringkasan_qc_sig.md`, `run_record_r-gis-001.json`, `peta_validasi_sig.png` |

---

## 1. Problem — Permasalahan

Sebelum data spasial vektor digunakan untuk analisis overlay, buffering, atau pemodelan GIS lanjutan, kita harus memastikan integritas topologi dan struktur datanya. Satu kesalahan geometri (misalnya *polygon self-intersection*, *null geometry*, koordinat ganda, atau ketidakcocokan Sistem Referensi Spasial/CRS) dapat menyebabkan kegagalan algoritma atau analisis spasial yang tidak akurat.

Recipe ini menjawab satu pertanyaan spesifik: **Apakah berkas data vektor ini layak digunakan sebagai input analisis SIG, dan fitur (*feature*) mana yang perlu diperbaiki terlebih dahulu?**

**Tujuan yang dapat diperiksa:**
1. Memeriksa keberadaan CRS dan ketersediaan atribut wajib (*schema check*).
2. Memeriksa validitas geometri sesuai standar OGC (*valid geometry*, *null geometry*, *self-intersection*).
3. **Menandai** (*flagging*) fitur bermasalah pada kolom atribut baru tanpa mengubah/menghapus *geometry* mentah.
4. Menjalankan verifikasi otomatis, lalu menyimpan laporan audit dan *run record*.

**Batasan keputusan:** Hasil recipe ini berupa daftar *flag* spasial. Fitur yang ter-flag harus diperiksa atau diperbaiki secara manual pada perangkat lunak GIS (seperti QGIS/ArcGIS) sebelum dihapus.

---

## 2. Minimal Theory — Teori Singkat

Kualitas data vektor GIS ditentukan oleh dua aspek utama:

1. **Sistem Referensi Koordinat (CRS):** Data spasial harus terdefinisi proyeksinya (misalnya EPSG:4326 / WGS84 atau EPSG:32750 / UTM Zone 50S) agar pengukuran luas dan jarak konsisten.
2. **Validitas Geometri (OGC Standard):** Suatu polygon dianggap valid jika ring pembentuknya tidak saling berpotongan (*self-intersection*) dan titik koordinatnya tidak berulang secara berurutan.

| Aturan QC | Ambang / Kondisi | Alasan Spasial / Fisis |
|---|---|---|
| CRS Tidak Terdefinisi | CRS = `None` | Lokasi spasial tidak dapat ditumpangtindihkan secara presisi |
| Geometri Kosong (*Null*) | `geometry.is_empty` atau `None` | Fitur atribut ada tanpa koordinat lokasi |
| Geometri Tidak Valid | `geometry.is_valid` = `False` | Berpotongan dengan diri sendiri (*self-intersection*), menyebabkan kegagalan analisis *overlay* |
| Atribut Wajib Kosong | Field bernilai `NaN` / *empty string* | Menghilangkan identitas wilayah/kategori atribut |

---

## 3. Input dan Constraint Contract

| Aspek | Ketentuan |
|---|---|
| **Input File** | Berkas GeoJSON / Shapefile (`data_vektor.geojson`) |
| **Kolom Wajib** | `id`, `nama_area`, `geometry` |
| **CRS Contract** | Harus terdefinisi. Jika tidak terdefinisi, minta klarifikasi dan jangan menebak otomatis. |
| **Boleh Diubah** | Hanya berkas keluaran baru. **Data mentah tidak boleh ditimpa/diubah** (dibuktikan dengan hash SHA-256 sebelum dan sesudah analisis). |
| **Dilarang** | Menghapus polygon/polyline yang bermasalah secara sepihak. |
| **Jaringan** | Tidak diperlukan (*Offline processing*). |

---

## 4. Prompt — Instruksi untuk AI (P-GIS-001)

> **R (Role):** Kamu adalah analis Sistem Informasi Geografis (SIG) yang teliti dalam pengelolaan data spasial.  
> **T (Task):** Baca `[NAMA_FILE_VEKTOR]`, periksa validitas geometri OGC dan kelengkapan atributnya, tandai fitur yang bermasalah pada kolom atribut baru, lalu susun laporan QA/QC spasial.  
> **C (Context):** Hasil audit digunakan untuk menentukan data mana yang layak masuk ke pemodelan SIG lanjutan. Ini bukan analisis layout peta.  
> **I (Input):** Berkas GeoJSON/Shapefile dengan kolom wajib `id`, `nama_area`, dan `geometry`. Jangan menggunakan data spasial lain.  
> **K (Constraints):** Jangan mengubah atau menghapus geometri mentah. Tandai dengan atribut `qc_flag` dan `qc_reason`. Tuliskan alasannya jika ada geometri *null*, *invalid/self-intersecting*, atau atribut kosong.  
> **O (Output):** Berkas GeoJSON ter-QC `batas_admin_qc.geojson`, laporan Markdown `ringkasan_qc_sig.md`, visualisasi `peta_validasi_sig.png`, dan `run_record_r-gis-001.json`.  
> **Q (Quality criteria):** Jumlah fitur keluaran = jumlah fitur masukan; setiap *flag* memiliki alasan yang jelas; hash data mentah tidak berubah.  
> **V (Verification):** Jalankan seluruh pemeriksaan skema dan geometri sebelum menyimpan hasil akhir. Jika ada yang gagal, hentikan proses dan laporkan hasilnya.

---

## 5. Code — Persiapan dan Pengaturan

```python
import sys, json, hashlib, datetime
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RECIPE_ID, PROMPT_ID = "R-GIS-001", "P-GIS-001"

# ---- Pengaturan File ----
RAW_FILE = Path("data_vektor.geojson")

# ---- Pencatat Verifikasi ----
CHECKS = []
def check(nama, kondisi, detail=""):
    CHECKS.append({"pemeriksaan": nama, "lulus": bool(kondisi), "detail": detail})
    print(("[LULUS] " if kondisi else "[GAGAL] ") + nama + (f"  ->  {detail}" if detail else ""))

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

print("Python", sys.version.split()[0], "| GeoPandas", gpd.__version__)